# Short Volshock BC Initialization Official Replay v1

Purpose: replay the `bc_initialization` candidate retained by `short_volshock_drawdown_drift_reward_short_ppo_screen_v1` under the official qlib warm-start metric path, then PK the ordinary fold backtest evidence against the official replay evidence.

Governance: diagnostic/model-selection loop only; no actor is retained or promoted from this notebook. Experiment-specific conclusions stay in this runtime card.

In [1]:
from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.constrained_checkpoint_selector import (
    score_constrained_checkpoints,
    summarize_constrained_checkpoint_selection,
)
from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "short_volshock_bc_initialization_official_replay_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_REWARD_CARD = ROOT / "runtime/reward/card/short_volshock_drawdown_drift_reward_short_ppo_screen_v1"
TARGET_CARD = ROOT / "runtime/model_selection/card/short_volshock_target_generation_handoff_v1"
TARGETS_PATH = TARGET_CARD / "short_volshock_de_risk_targets.csv"
FRAME_PATH = ROOT / "data/soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
SHORT_VOLSHOCK_SLICE_ACCURACY_MIN = 0.60
SHORT_VOLSHOCK_SLICE_SAMPLES_MIN = 20
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

WALK_FORWARD_WINDOWS = {
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://doi.org/10.21314/JCF.2016.322",
]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def fold_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    train_end = pd.Timestamp(spec["train"][1])
    subset = targets[
        targets["fold"].astype(str).eq(str(spec["target_fold"]))
        & (pd.to_datetime(targets["datetime"]) <= train_end)
    ].copy()
    return subset.sort_values("datetime").reset_index(drop=True)


def gate_pass(relative_return: float, max_dd: float) -> bool:
    return bool(float(relative_return) >= TARGET_RELATIVE_RETURN and float(max_dd) <= TARGET_MAX_DD)


def gate_loss(relative_return: float, max_dd: float) -> float:
    return float(max(0.0, TARGET_RELATIVE_RETURN - float(relative_return)) + max(0.0, float(max_dd) - TARGET_MAX_DD) * 10.0)


def max_drawdown_interval_from_daily(path: Path, *, candidate: str, fold: str) -> dict[str, Any]:
    daily = pd.read_csv(path, parse_dates=["datetime"])
    nav = pd.to_numeric(daily["strategy_nav"], errors="coerce").ffill()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    if daily.empty:
        return {"candidate": candidate, "fold": fold, "path": str(path.relative_to(ROOT)), "max_dd": math.nan}
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    interval = daily.loc[peak_idx:trough_idx].copy()
    action_col = daily.get("action", pd.Series("", index=daily.index)).astype(str).str.upper()
    return {
        "candidate": candidate,
        "fold": fold,
        "path": str(path.relative_to(ROOT)),
        "peak_date": str(pd.Timestamp(daily.loc[peak_idx, "datetime"]).date()),
        "trough_date": str(pd.Timestamp(daily.loc[trough_idx, "datetime"]).date()),
        "max_dd": float(abs(dd.min()) * 100.0),
        "interval_days": int(len(interval)),
        "interval_avg_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).mean()),
        "interval_min_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).min()),
        "interval_max_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).max()),
        "interval_trades": int(action_col.loc[peak_idx:trough_idx].isin(["BUY", "SELL"]).sum()),
    }


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
source_selection = load_json(SOURCE_REWARD_CARD / "selection.json")
source_summary = load_json(SOURCE_REWARD_CARD / "summary.json")
source_fold_results = pd.read_csv(SOURCE_REWARD_CARD / "ppo_fold_results.csv")
reward_patch = dict(source_selection.get("reward_patch", {}))
profile = load_profile(PROFILE_NAME)
# The reward patch is included to mirror the source notebook configuration. BC fit and official replay do not consume reward terms.
if reward_patch:
    profile = replace(profile, reward=replace(profile.reward, **reward_patch))
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
targets = pd.read_csv(TARGETS_PATH, parse_dates=["datetime"])
frame = pd.read_pickle(FRAME_PATH)

coverage_rows: list[dict[str, Any]] = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    fold_targets = fold_subset(targets, spec)
    short_target = fold_targets.get("short_volshock_target_state", pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool)
    coverage_rows.append(
        {
            "fold": fold_name,
            "target_fold": spec["target_fold"],
            "target_rows": int(len(fold_targets)),
            "unique_target_dates": int(fold_targets["datetime"].nunique()) if len(fold_targets) else 0,
            "short_volshock_target_rows": int(short_target.sum()),
            "target_start": str(fold_targets["datetime"].min().date()) if len(fold_targets) else "",
            "target_end": str(fold_targets["datetime"].max().date()) if len(fold_targets) else "",
            "train_start": spec["train"][0],
            "train_end": spec["train"][1],
            "valid_start": spec["valid"][0],
            "valid_end": spec["valid"][1],
            "test_start": spec["test"][0],
            "test_end": spec["test"][1],
        }
    )

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / "coverage.csv", index=False)
write_json(
    CARD / "coverage.json",
    {
        "experiment": EXPERIMENT,
        "profile": PROFILE_NAME,
        "source_experiment": "short_volshock_drawdown_drift_reward_short_ppo_screen_v1",
        "source_decision": source_selection.get("decision"),
        "source_constrained_selector_selected_candidate": source_selection.get("constrained_selector_selected_candidate"),
        "source_next_mechanism": source_selection.get("next_mechanism"),
        "reward_patch": reward_patch,
        "targets_path": str(TARGETS_PATH.relative_to(ROOT)),
        "feature_frame_path": str(FRAME_PATH.relative_to(ROOT)),
        "windows": coverage_rows,
        "skipped_windows": [
            {
                "fold": "wf_2020",
                "reason": "short-volshock targets begin in 2020, so a wf_2020 training target would leak that test year",
            }
        ],
    },
)
print({
    "experiment": EXPERIMENT,
    "profile": PROFILE_NAME,
    "source_decision": source_selection.get("decision"),
    "source_selected": source_selection.get("constrained_selector_selected_candidate"),
    "frame_rows": int(len(frame)),
    "targets": int(len(targets)),
})
display(coverage)


{'experiment': 'short_volshock_bc_initialization_official_replay_v1', 'profile': 'sparse_release_bc_ppo_screen_v1', 'source_decision': 'short_volshock_reward_ppo_drift_failed_no_promotion', 'source_selected': 'bc_initialization', 'frame_rows': 2973, 'targets': 1009}


,fold,target_fold,target_rows,unique_target_dates,short_volshock_target_rows,target_start,target_end,train_start,train_end,valid_start,valid_end,test_start,test_end
0,wf_2022,wf_2022_train,253,253,133,2020-01-02,2020-12-31,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
1,wf_2023_2024,wf_2023_2024_train,756,756,336,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


## Official Replay

The actor fitting call mirrors the source `bc_initialization` branch: same profile, same fold-local targets, same seed, plain BC, `sample_weight_config=None`, and `sample_size=0`. The replay changes only the metric surface: qlib gets a warm-start segment and metrics are sliced back to the requested held-out fold.

In [3]:
init_qlib(QLIB_DATA_PATH)

fold_cache: dict[str, dict[str, Any]] = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec["test"], lookback=LOOKBACK)
    fold_cache[fold_name] = {"dataset": dataset, "feature_full": feature_full, "price_full": price_full, "warm_seg": warm_seg}

warm_segments = pd.DataFrame(
    [
        {
            "fold": fold,
            "feature_start": str(cache["feature_full"].index.min().date()),
            "feature_end": str(cache["feature_full"].index.max().date()),
            "test_start": WALK_FORWARD_WINDOWS[fold]["test"][0],
            "test_end": WALK_FORWARD_WINDOWS[fold]["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
            "lookback": int(LOOKBACK),
        }
        for fold, cache in fold_cache.items()
    ]
)
warm_segments.to_csv(CARD / "warm_start_segments.csv", index=False)

official_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
daily_manifest: list[dict[str, str]] = []
interval_rows: list[dict[str, Any]] = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"\n[official:{fold_name}] train={spec['train']} valid={spec['valid']} test={spec['test']}")
    fold_targets = fold_subset(targets, spec)
    if fold_targets.empty:
        official_rows.append({"candidate": "bc_initialization", "fold": fold_name, "status": "skipped_no_targets"})
        continue

    set_seed(SEED)
    fit = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=BC_TRAIN_CONFIG,
        sample_weight_config=None,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    slice_metrics = evaluate_bc_actor_slice_fit(
        fit.actor,
        fit.obs,
        fit.actions,
        bc_dataset.metadata,
        slice_cols=("short_volshock_target_state", "short_volshock_state", "risk_state", "is_downtrend_state", "is_rebound_cluster"),
    )
    slice_summary = summarize_bc_slice_fit(
        slice_metrics,
        required_slices=("short_volshock_target_state",),
        min_accuracy=SHORT_VOLSHOCK_SLICE_ACCURACY_MIN,
        min_samples=SHORT_VOLSHOCK_SLICE_SAMPLES_MIN,
    )
    tagged_slices = slice_metrics.copy()
    tagged_slices.insert(0, "fold", fold_name)
    tagged_slices.insert(0, "candidate", "bc_initialization")
    slice_frames.append(tagged_slices)
    dist = fit.action_distribution.copy()
    dist.insert(0, "fold", fold_name)
    dist.insert(0, "candidate", "bc_initialization")
    action_frames.append(dist)

    short_row = slice_metrics.set_index("slice").loc["short_volshock_target_state"].to_dict()
    fit_rows.append(
        {
            "candidate": "bc_initialization",
            "fold": fold_name,
            "target_rows": int(len(fold_targets)),
            "bc_samples": int(len(fit.actions)),
            "before_accuracy": float(fit.metrics.get("before_accuracy", math.nan)),
            "after_accuracy": float(fit.metrics.get("after_accuracy", math.nan)),
            "after_loss": float(fit.metrics.get("after_loss", math.nan)),
            "short_volshock_target_samples": int(short_row.get("samples", 0)),
            "short_volshock_target_accuracy": float(short_row.get("accuracy", math.nan)),
            "slice_gate": bool(slice_summary["slice_fit_gate"]),
            "slice_failed_checks": json.dumps(slice_summary.get("failed_checks", []), sort_keys=True),
        }
    )

    cache = fold_cache[fold_name]
    bt = backtest_profile(
        fit.actor,
        cache["feature_full"],
        cache["price_full"],
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=cache["warm_seg"],
    )
    warm_daily = slice_warm_start_daily_path(
        portfolio_df=bt["portfolio"],
        price_series=cache["price_full"],
        eval_seg=spec["test"],
        position_history=bt.get("position_history", []),
        trade_signals=bt.get("trade_signals", []),
    )
    daily_path = CARD / f"bc_initialization_{fold_name}_official_daily_path.csv"
    warm_daily.to_csv(daily_path, index=False)
    daily_manifest.append({"candidate": "bc_initialization", "fold": fold_name, "path": str(daily_path.relative_to(ROOT))})
    interval_rows.append(max_drawdown_interval_from_daily(daily_path, candidate="bc_initialization", fold=fold_name))
    metrics = daily_path_metrics(warm_daily)
    full_gate = gate_pass(float(metrics["relative_return"]), float(metrics["max_dd"]))
    official_rows.append(
        {
            "candidate": "bc_initialization",
            "fold": fold_name,
            "status": "evaluated",
            "requested_start": spec["test"][0],
            "requested_end": spec["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
            "official_warm_return": float(metrics["return"]),
            "official_warm_buyhold_return": float(metrics["buy_and_hold_return"]),
            "official_warm_relative_return": float(metrics["relative_return"]),
            "official_warm_max_dd": float(metrics["max_dd"]),
            "official_warm_avg_pos": float(metrics["avg_pos"]),
            "trades": int(bt.get("trades", 0)),
            "slice_gate": bool(slice_summary["slice_fit_gate"]),
            "window_gate_pass": bool(full_gate and slice_summary["slice_fit_gate"]),
            "gate_loss": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"])) + (0.0 if slice_summary["slice_fit_gate"] else 20.0),
            "daily_path": str(daily_path.relative_to(ROOT)),
        }
    )

fit_metrics = pd.DataFrame(fit_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
bc_slice_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
official_metrics = pd.DataFrame(official_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
drawdown_intervals = pd.DataFrame(interval_rows).sort_values(["fold", "candidate"]).reset_index(drop=True) if interval_rows else pd.DataFrame()

fit_metrics.to_csv(CARD / "bc_fit_metrics.csv", index=False)
bc_slice_metrics.to_csv(CARD / "bc_slice_metrics.csv", index=False)
action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)
official_metrics.to_csv(CARD / "official_replay_metrics.csv", index=False)
pd.DataFrame(daily_manifest).to_csv(CARD / "daily_manifest.csv", index=False)
drawdown_intervals.to_csv(CARD / "drawdown_interval_diagnostic.csv", index=False)

display(warm_segments)
display(fit_metrics)
display(official_metrics)
display(drawdown_intervals)


[61593:MainThread](2026-05-27 00:51:01,765) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61593:MainThread](2026-05-27 00:51:02,634) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61593:MainThread](2026-05-27 00:51:02,637) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[61593:MainThread](2026-05-27 00:51:04,772) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[61593:MainThread](2026-05-27 00:51:04,773) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,fold,feature_start,feature_end,test_start,test_end,warm_start,warm_end,lookback
0,wf_2022,2011-03-11,2022-12-30,2022-01-01,2022-12-31,2021-10-07,2022-12-30,60
1,wf_2023_2024,2011-03-11,2024-12-31,2024-01-01,2024-12-31,2023-10-05,2024-12-31,60


,candidate,fold,target_rows,bc_samples,before_accuracy,after_accuracy,after_loss,short_volshock_target_samples,short_volshock_target_accuracy,slice_gate,slice_failed_checks
0,bc_initialization,wf_2022,253,253,0.272727,0.924901,0.979606,133,1.000000,True,[]
1,bc_initialization,wf_2023_2024,756,755,0.181457,0.772185,1.135113,335,0.925373,True,[]


,candidate,fold,status,requested_start,requested_end,warm_start,warm_end,official_warm_return,official_warm_buyhold_return,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,trades,slice_gate,window_gate_pass,gate_loss,daily_path
0,bc_initialization,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-18.963676,-86.588072,67.624396,22.261989,0.097005,42,True,False,72.619890,runtime/model_selection/card/short_volshock_bc...
1,bc_initialization,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,4.662264,-0.356635,5.018899,16.693087,0.183114,49,True,False,41.911976,runtime/model_selection/card/short_volshock_bc...


,candidate,fold,path,peak_date,trough_date,max_dd,interval_days,interval_avg_pos,interval_min_pos,interval_max_pos,interval_trades
0,bc_initialization,wf_2022,runtime/model_selection/card/short_volshock_bc...,2022-01-03,2022-11-03,22.261989,212,0.086226,0.000000,0.459725,27
1,bc_initialization,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,2024-03-07,2024-11-15,16.693087,177,0.140472,0.043714,0.533556,28


## PK: Source Fold Backtest vs Official Replay

The source reward notebook selected `bc_initialization` from ordinary held-out fold backtests. This section compares those numbers against the official warm-start replay metrics used for promotion-style review.

In [4]:
source_bc_rows = []
for row in source_fold_results.to_dict("records"):
    if str(row.get("status")) != "evaluated":
        continue
    source_bc_rows.append(
        {
            "candidate": "bc_initialization",
            "fold": row["fold"],
            "source_bc_relative_return": float(row.get("bc_test_relative_return", math.nan)),
            "source_bc_max_dd": float(row.get("bc_test_max_dd", math.nan)),
            "source_bc_return": float(row.get("bc_test_return", math.nan)),
            "source_bc_buyhold_return": float(row.get("bc_test_buy_and_hold_return", math.nan)),
            "source_bc_avg_pos": float(row.get("bc_test_avg_pos", math.nan)),
            "source_bc_gate": bool(row.get("bc_gate", False)),
        }
    )
source_bc = pd.DataFrame(source_bc_rows)
pk = official_metrics.merge(source_bc, on=["candidate", "fold"], how="left")
pk["official_gate"] = pk["window_gate_pass"].astype(bool)
pk["relative_delta_official_minus_source"] = pk["official_warm_relative_return"] - pk["source_bc_relative_return"]
pk["maxdd_delta_official_minus_source"] = pk["official_warm_max_dd"] - pk["source_bc_max_dd"]
pk["gate_flip"] = pk["source_bc_gate"].astype(bool) != pk["official_gate"].astype(bool)
pk = pk[
    [
        "candidate",
        "fold",
        "source_bc_gate",
        "official_gate",
        "gate_flip",
        "source_bc_relative_return",
        "official_warm_relative_return",
        "relative_delta_official_minus_source",
        "source_bc_max_dd",
        "official_warm_max_dd",
        "maxdd_delta_official_minus_source",
        "source_bc_avg_pos",
        "official_warm_avg_pos",
        "trades",
        "slice_gate",
        "daily_path",
    ]
]
pk.to_csv(CARD / "source_vs_official_pk.csv", index=False)

gate_metrics = official_metrics[official_metrics["status"].eq("evaluated")].rename(
    columns={"fold": "window", "official_warm_relative_return": "relative_return", "official_warm_max_dd": "max_dd"}
)[["candidate", "window", "relative_return", "max_dd"]]
constraint_scores = score_constrained_checkpoints(gate_metrics) if not gate_metrics.empty else pd.DataFrame()
constraint_summary = summarize_constrained_checkpoint_selection(constraint_scores)
constraint_scores.to_csv(CARD / "constrained_checkpoint_scores.csv", index=False)

all_official_gate = bool((not pk.empty) and pk["official_gate"].all() and len(pk) == len(WALK_FORWARD_WINDOWS))
any_gate_flip = bool((not pk.empty) and pk["gate_flip"].any())
source_all_gate = bool((not pk.empty) and pk["source_bc_gate"].all() and len(pk) == len(WALK_FORWARD_WINDOWS))
max_official_dd = float(pd.to_numeric(pk["official_warm_max_dd"], errors="coerce").max()) if not pk.empty else math.nan
min_official_rel = float(pd.to_numeric(pk["official_warm_relative_return"], errors="coerce").min()) if not pk.empty else math.nan
if all_official_gate:
    decision = "short_volshock_bc_initialization_official_replay_passed_partial_wf_no_promotion"
    next_mechanism = "run wf_2020-compatible fixed-split/full replay and decide whether to retain a reproducible BC actor bundle"
elif source_all_gate and any_gate_flip:
    decision = "short_volshock_bc_initialization_official_replay_gate_flip_no_promotion"
    next_mechanism = "treat ordinary fold backtest as optimistic; repair metric alignment before PPO or actor retention"
else:
    decision = "short_volshock_bc_initialization_official_replay_failed_no_promotion"
    next_mechanism = "drop this BC initialization as a promotion path and return to target/official-replay alignment"

conclusion = []
for row in pk.sort_values("fold").to_dict("records"):
    conclusion.append(
        f"{row['fold']}: source gate={bool(row['source_bc_gate'])}, official gate={bool(row['official_gate'])}, "
        f"source rel={row['source_bc_relative_return']:.2f}pp vs official rel={row['official_warm_relative_return']:.2f}pp, "
        f"source MaxDD={row['source_bc_max_dd']:.2f}% vs official MaxDD={row['official_warm_max_dd']:.2f}%, "
        f"gate_flip={bool(row['gate_flip'])}."
    )
for row in drawdown_intervals.to_dict("records"):
    conclusion.append(
        f"{row['fold']} official drawdown interval: {row.get('peak_date', '')} to {row.get('trough_date', '')}, "
        f"MaxDD={row.get('max_dd', math.nan):.2f}%, avg_pos={row.get('interval_avg_pos', math.nan):.3f}."
    )
conclusion.append("No actor retention or promotion from this card; the replay is only a diagnostic PK against the source fold backtest.")
conclusion.append(f"Next mechanism: {next_mechanism}.")

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "profile": PROFILE_NAME,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "short_volshock_drawdown_drift_reward_short_ppo_screen_v1/bc_initialization",
    "changed_layer": ["model_selection", "metric_reconciliation", "bc_warm_start"],
    "closest_historical_success_case": "short_volshock_reward_ppo_screen found bc_initialization cleared ordinary wf_2022 and wf_2023_2024 constraints",
    "closest_historical_failure_case": "short_volshock_bc_checkpoint_alignment_v1 showed official warm-start replay could invalidate ordinary short-volshock BC gate evidence",
    "duplicate_check": "This run does not tune the target, reward, or PPO; it replays only the retained bc_initialization candidate under the official warm-start metric path and PKs source-vs-official metrics.",
    "source_experiment": "short_volshock_drawdown_drift_reward_short_ppo_screen_v1",
    "source_decision": source_selection.get("decision"),
    "source_next_mechanism": source_selection.get("next_mechanism"),
    "selected_candidate": "bc_initialization",
    "source_all_gate": source_all_gate,
    "official_all_gate": all_official_gate,
    "any_gate_flip": any_gate_flip,
    "selector_summary": constraint_summary,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "min_official_relative_return": min_official_rel,
    "max_official_max_dd": max_official_dd,
    "folds_evaluated": list(WALK_FORWARD_WINDOWS.keys()),
    "wf_2020_coverage": False,
    "wf_2020_skip_reason": "short-volshock targets begin in 2020; using them would leak the wf_2020 test year",
    "promotion_allowed": False,
    "actor_retention": "no actor bundle retained; rerun notebook to regenerate the diagnostic BC actor",
    "promotion_blocker": "partial walk-forward official replay only; no wf_2020, no full fixed-split proof, and no retained actor bundle",
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "coverage.csv",
        "coverage.json",
        "warm_start_segments.csv",
        "bc_fit_metrics.csv",
        "bc_slice_metrics.csv",
        "bc_action_distribution.csv",
        "official_replay_metrics.csv",
        "daily_manifest.csv",
        "drawdown_interval_diagnostic.csv",
        "source_vs_official_pk.csv",
        "constrained_checkpoint_scores.csv",
        "selection.json",
        "summary.json",
        "*_official_daily_path.csv",
    ],
}
summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "promotion_allowed": False,
    "source_vs_official_pk": pk.to_dict("records"),
    "official_metrics": official_metrics.to_dict("records"),
    "fit_metrics": fit_metrics.to_dict("records"),
    "selector_summary": constraint_summary,
    "drawdown_interval_diagnostic": drawdown_intervals.to_dict("records"),
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)

display(pk)
display(constraint_scores)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


,candidate,fold,source_bc_gate,official_gate,gate_flip,source_bc_relative_return,official_warm_relative_return,relative_delta_official_minus_source,source_bc_max_dd,official_warm_max_dd,maxdd_delta_official_minus_source,source_bc_avg_pos,official_warm_avg_pos,trades,slice_gate,daily_path
0,bc_initialization,wf_2022,True,False,True,66.727389,67.624396,0.897006,14.997907,22.261989,7.264082,0.119868,0.097005,42,True,runtime/model_selection/card/short_volshock_bc...
1,bc_initialization,wf_2023_2024,True,False,True,31.124073,5.018899,-26.105174,13.361110,16.693087,3.331977,0.138743,0.183114,49,True,runtime/model_selection/card/short_volshock_bc...


,candidate,selector_gate_pass,decision,constrained_selector_loss,constraint_gap_loss,failed_window_loss,drift_loss,score_loss,window_count,gate_pass_count,...,min_relative_return,max_max_dd,relative_drift_gap,max_dd_drift_gap,alignment_loss,checkpoint_objective_loss,missing_target_ratio,coverage_gate,target_relative_return_pp,target_max_dd_pct
0,bc_initialization,False,constrained_checkpoint_no_promotion,314.531866,114.531866,200.0,0.0,0.0,2,0,...,5.018899,22.261989,0.0,0.0,0.0,0.0,0.0,True,30.0,15.0


{
  "experiment": "short_volshock_bc_initialization_official_replay_v1",
  "created_at": "2026-05-26T16:51:30.501566+00:00",
  "decision": "short_volshock_bc_initialization_official_replay_gate_flip_no_promotion",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "short_volshock_drawdown_drift_reward_short_ppo_screen_v1/bc_initialization",
  "changed_layer": [
    "model_selection",
    "metric_reconciliation",
    "bc_warm_start"
  ],
  "closest_historical_success_case": "short_volshock_reward_ppo_screen found bc_initialization cleared ordinary wf_2022 and wf_2023_2024 constraints",
  "closest_historical_failure_case": "short_volshock_bc_checkpoint_alignment_v1 showed official warm-start replay could invalidate ordinary short-volshock BC gate evidence",
  "duplicate_check": "This run does not tune the target, reward, or PPO; it replays only the retained bc_initializatio

## Conclusion

The final decision is written to `selection.json` and `summary.json`; the notebook output above is the single-run conclusion for this runtime card.

In [5]:
summary = load_json(CARD / "summary.json")
lines = ["# Runtime Conclusion", ""]
lines.append(f"- decision: `{summary['decision']}`")
lines.append(f"- next: `{summary['next_mechanism']}`")
lines.append("- promotion_allowed: `False`")
lines.append("")
for item in summary.get("conclusion", []):
    lines.append(f"- {item}")
display(Markdown("\n".join(lines)))


# Runtime Conclusion

- decision: `short_volshock_bc_initialization_official_replay_gate_flip_no_promotion`
- next: `treat ordinary fold backtest as optimistic; repair metric alignment before PPO or actor retention`
- promotion_allowed: `False`

- wf_2022: source gate=True, official gate=False, source rel=66.73pp vs official rel=67.62pp, source MaxDD=15.00% vs official MaxDD=22.26%, gate_flip=True.
- wf_2023_2024: source gate=True, official gate=False, source rel=31.12pp vs official rel=5.02pp, source MaxDD=13.36% vs official MaxDD=16.69%, gate_flip=True.
- wf_2022 official drawdown interval: 2022-01-03 to 2022-11-03, MaxDD=22.26%, avg_pos=0.086.
- wf_2023_2024 official drawdown interval: 2024-03-07 to 2024-11-15, MaxDD=16.69%, avg_pos=0.140.
- No actor retention or promotion from this card; the replay is only a diagnostic PK against the source fold backtest.
- Next mechanism: treat ordinary fold backtest as optimistic; repair metric alignment before PPO or actor retention.